# Reproducing the Q3 results: Bangla cyclone and flood response line

This notebook reproduces **every number claimed in Answer 3** from scratch. It is fully self-contained: the cells below write the prototype's code, run it, and then check each claim automatically.

* **Stage 1:** M2M self-play generates 1,000 **synthetic** dialogues, which are checked against six safety invariants; the checker itself is validated by mutation testing.
* **Stage 2:** persona stress testing compares NLU v1 (standard Bangla only) with v2 (extended lexicon) across four caller registers.

Requirements: Python 3 standard library only. The random seed is fixed (2026), so results are identical on every run.

**Run:** *Runtime → Run all* (under a minute).

## 1. Write the prototype code

In [1]:
%%writefile selfplay.py
"""Miniature M2M self-play for a Bangla cyclone and flood response line.

Components follow Shah et al. (2018): a task schema and API supplied by the
developer, a scenario sampler (goal + profile), an agenda-based user
simulator and a finite-state system agent. Dialogues are emitted as
dialogue-act outlines with Bangla templates.
"""
import json
import random

UPAZILAS = ["কয়রা", "শ্যামনগর", "কলাপাড়া", "হাতিয়া", "কুতুবদিয়া"]
VULNERABLE = ["বয়স্ক", "শিশু", "অন্তঃসত্ত্বা", "প্রতিবন্ধী", "নেই"]
LANDMARKS = ["বড় মসজিদের পাশে", "প্রাইমারি স্কুলের কাছে", "বাজারের পাশে"]

# Mock shelter database: upazila -> [(shelter, remaining capacity)].
# Kutubdia is deliberately full, creating unsatisfiable shelter requests.
SHELTERS = {
    "কয়রা": [("কয়রা সদর আশ্রয়কেন্দ্র", 40), ("মহারাজপুর আশ্রয়কেন্দ্র", 6)],
    "শ্যামনগর": [("গাবুরা আশ্রয়কেন্দ্র", 12)],
    "কলাপাড়া": [("কলাপাড়া হাই স্কুল আশ্রয়কেন্দ্র", 25), ("ধানখালী আশ্রয়কেন্দ্র", 3)],
    "হাতিয়া": [("হাতিয়া সদর আশ্রয়কেন্দ্র", 8)],
    "কুতুবদিয়া": [("কুতুবদিয়া আশ্রয়কেন্দ্র", 0)],
}


def api_find_shelter(upazila, size):
    """First shelter in the upazila with enough remaining capacity, else None."""
    for name, cap in SHELTERS.get(upazila, []):
        if cap >= size:
            return name
    return None


def required_slots(state):
    status = state.get("status")
    if status is None:
        return ["status"]
    if status == "safe":
        return []
    slots = ["location"]
    if state.get("location") == "অজানা":
        slots.append("landmark")
    slots.append("household_size")
    if status == "rescue":
        slots.append("vulnerable")
    return slots


# ----------------------------------------------------------- scenario --
def sample_scenario(rng):
    status = rng.choices(["safe", "shelter", "rescue"], weights=[0.3, 0.4, 0.3])[0]
    goal = {"status": status}
    if status != "safe":
        goal["location"] = "অজানা" if rng.random() < 0.1 else rng.choice(UPAZILAS)
        if goal["location"] == "অজানা":
            goal["landmark"] = rng.choice(LANDMARKS)
        goal["household_size"] = rng.randint(1, 8)
        goal["vulnerable"] = rng.choices(VULNERABLE, weights=[2, 2, 1, 1, 4])[0]
        goal["can_travel"] = rng.random() > 0.3
    profile = {"verbosity": rng.choice([1, 2, 3])}
    return goal, profile


# ------------------------------------------------------ user simulator --
class UserSim:
    def __init__(self, goal, profile, rng):
        self.goal, self.profile, self.rng = goal, profile, rng
        self.agenda = [s for s in ("status", "location", "landmark", "household_size", "vulnerable")
                       if s in goal]
        rest = self.agenda[1:]
        rng.shuffle(rest)                            # status first; the rest in any order
        self.agenda = self.agenda[:1] + rest

    def _inform(self, k):
        out = []
        while self.agenda and len(out) < k:
            s = self.agenda.pop(0)
            out.append(("inform", s, self.goal[s]))
        return out

    def respond(self, sys_acts):
        a, s, _ = sys_acts[-1]
        if a == "request":
            if s in self.agenda:
                self.agenda.remove(s)
            return [("inform", s, self.goal[s])] + self._inform(self.profile["verbosity"] - 1)
        if a == "offer":
            return [("affirm", None, None)] if self.goal["can_travel"] else [("negate", "travel", None)]
        return [("thank_you", None, None), ("bye", None, None)]


# -------------------------------------------------------- system agent --
class SystemFSM:
    def __init__(self):
        self.state = {}

    def _priority(self):
        return "high" if self.state.get("vulnerable") not in (None, "নেই") else "normal"

    def act(self, user_acts):
        for a, s, v in user_acts:
            if a == "inform":
                self.state[s] = v
        last = user_acts[-1][0]
        if last == "bye":
            return [("bye", None, None)]
        if last == "affirm":
            return [("notify_success", "shelter", self.state["offered"])]
        if last == "negate":                      # cannot reach shelter -> becomes rescue
            self.state["status"] = "rescue"
        missing = [s for s in required_slots(self.state) if s not in self.state]
        if missing:
            return [("request", missing[0], None)]
        if self.state["status"] == "safe":
            return [("inform", "advice", "stay_alert")]
        if self.state["status"] == "rescue":
            return [("escalate_rescue", "priority", self._priority())]
        if self.state.get("location") == "অজানা":
            return [("escalate_officer", "reason", "location_unknown")]
        shelter = api_find_shelter(self.state.get("location"), self.state["household_size"])
        if shelter is None:
            return [("escalate_officer", "reason", "no_capacity")]
        self.state["offered"] = shelter
        return [("offer", "shelter", shelter)]


END_ACTS = ("bye", "escalate_rescue", "escalate_officer", "escalate_human")


def self_play(rng, max_turns=20):
    goal, profile = sample_scenario(rng)
    user, system = UserSim(goal, profile, rng), SystemFSM()
    turns = [("S", [("greeting", None, None), ("request", "status", None)])]
    u = user.respond(turns[0][1])
    while len(turns) < max_turns:
        turns.append(("U", u))
        s = system.act(u)
        turns.append(("S", s))
        if s[-1][0] in END_ACTS or u[-1][0] == "bye":
            break
        u = user.respond(s)
    if turns[-1][1][-1][0] != "bye":
        turns.append(("U", [("bye", None, None)]))
    return {"goal": goal, "profile": profile,
            "turns": [{"speaker": sp, "acts": [list(a) for a in acts]} for sp, acts in turns]}


def scripted_baseline(rng):
    """Happy path: rescue request, one slot per turn, always escalated."""
    turns = [("S", [("greeting", None, None), ("request", "status", None)]),
             ("U", [("inform", "status", "rescue")])]
    for s in ("location", "household_size", "vulnerable"):
        turns += [("S", [("request", s, None)]), ("U", [("inform", s, "x")])]
    turns += [("S", [("escalate_rescue", "priority", "normal")]), ("U", [("bye", None, None)])]
    return {"turns": [{"speaker": sp, "acts": [list(a) for a in acts]} for sp, acts in turns]}


if __name__ == "__main__":
    rng = random.Random(2026)
    outlines = [self_play(rng) for _ in range(1000)]
    with open("outlines.jsonl", "w", encoding="utf-8") as f:
        for o in outlines:
            f.write(json.dumps(o, ensure_ascii=False) + "\n")
    print("wrote", len(outlines), "outlines")


Writing selfplay.py


In [2]:
%%writefile evaluate_outlines.py
"""Validate cyclone-line outlines and compare diversity with a scripted baseline."""
import collections
import json
import random

from selfplay import SHELTERS, api_find_shelter, scripted_baseline

RESOLUTIONS = ("inform", "notify_success", "escalate_rescue", "escalate_officer")


def signature(turn):
    return tuple((a, s if a in ("inform", "request") else None) for a, s, _ in turn["acts"])


def diversity(dialogues, k=3):
    total = sum(len(d["turns"]) for d in dialogues)
    trans, subs, n_subs, outlines = set(), set(), 0, set()
    for d in dialogues:
        sig = [signature(t) for t in d["turns"]]
        trans.update(zip(sig, sig[1:]))
        for i in range(len(sig) - k):
            subs.add(tuple(sig[i:i + k + 1])); n_subs += 1
        outlines.add(tuple(sig))
    return {"unique_transitions_per_turn": len(trans) / total,
            "unique_subdialogues_k3": len(subs) / max(n_subs, 1),
            "unique_outlines_ratio": len(outlines) / len(dialogues)}


def check_invariants(d):
    g, errors, informed = d["goal"], [], set()
    acts = [(t["speaker"], a, s, v) for t in d["turns"] for a, s, v in t["acts"]]
    for sp, a, s, v in acts:
        if sp == "U" and a == "inform":
            informed.add(s)
        if sp == "S" and a == "request" and s in informed:
            errors.append(f"I1 re-requested {s}")
        if sp == "S" and a == "offer":
            caps = dict(SHELTERS.get(g.get("location"), []))
            if v not in caps or caps[v] < g["household_size"]:
                errors.append("I2 invalid shelter offer")
    sys_acts = [a for sp, a, _, _ in acts if sp == "S"]
    needs_rescue = g["status"] == "rescue" or ("offer" in sys_acts and not g.get("can_travel", True))
    if needs_rescue and ("escalate_rescue" not in sys_acts or "notify_success" in sys_acts):
        errors.append("I3 rescue need not escalated")
    for sp, a, s, v in acts:
        if a == "escalate_rescue":
            want = "high" if g.get("vulnerable") not in (None, "নেই") else "normal"
            if v != want:
                errors.append("I4 wrong priority")
    if g["status"] == "shelter" and g.get("location") != "অজানা":
        if ("escalate_officer" in sys_acts) != (api_find_shelter(g["location"], g["household_size"]) is None):
            errors.append("I5 capacity handling wrong")
    if acts[-1][1] != "bye" or not any(a in RESOLUTIONS for a in sys_acts):
        errors.append("I6 caller left unresolved")
    return errors


def outcome(d):
    sys_acts = [a for t in d["turns"] if t["speaker"] == "S" for a, _, _ in t["acts"]]
    if "escalate_rescue" in sys_acts:
        return "rescue_after_shelter_refusal" if "offer" in sys_acts else "rescue_escalated"
    if "escalate_officer" in sys_acts:
        reason = [v for t in d["turns"] for a, _, v in t["acts"] if a == "escalate_officer"][0]
        return "location_unknown" if reason == "location_unknown" else "no_shelter_capacity"
    if "notify_success" in sys_acts:
        return "shelter_confirmed"
    return "safe_advice"


if __name__ == "__main__":
    outlines = [json.loads(l) for l in open("outlines.jsonl", encoding="utf-8")]
    violations = [(i, e) for i, d in enumerate(outlines) for e in check_invariants(d)]
    print(f"Invariant violations: {len(violations)} across {len(outlines)} outlines")
    for v in violations[:8]:
        print("  ", v)
    print("\nOutcome distribution:")
    for k, v in collections.Counter(outcome(d) for d in outlines).most_common():
        print(f"  {k:<30}{v:>5}  ({v / len(outlines):.1%})")
    lengths = [len(d["turns"]) for d in outlines]
    print(f"\nTurns per dialogue: mean {sum(lengths)/len(lengths):.2f}, min {min(lengths)}, max {max(lengths)}")
    rng = random.Random(2026)
    base = [scripted_baseline(rng) for _ in range(len(outlines))]
    sp, bl = diversity(outlines), diversity(base)
    print("\nDiversity (self-play vs scripted):")
    for k in sp:
        print(f"  {k:<30} {sp[k]:.4f}   {bl[k]:.4f}")


Writing evaluate_outlines.py


In [3]:
%%writefile mutation_test.py
"""Inject one known fault per invariant; the checker must flag every mutant."""
import copy, json
from evaluate_outlines import check_invariants, outcome

outs = [json.loads(l) for l in open("outlines.jsonl", encoding="utf-8")]
pick = {}
for d in outs:
    pick.setdefault(outcome(d), d)

def acts_of(d):
    return [a for t in d["turns"] for a in t["acts"]]

def mutant(kind):
    if kind == "I1 re-request":
        d = copy.deepcopy(pick["rescue_escalated"])
        d["turns"].insert(-2, {"speaker": "S", "acts": [["request", "location", None]]})
    elif kind == "I2 invalid shelter":
        d = copy.deepcopy(pick["shelter_confirmed"])
        for a in acts_of(d):
            if a[0] == "offer": a[2] = "অস্তিত্বহীন আশ্রয়কেন্দ্র"
    elif kind == "I3 rescue dropped":
        d = copy.deepcopy(pick["rescue_escalated"])
        for t in d["turns"]:
            t["acts"] = [a for a in t["acts"] if a[0] != "escalate_rescue"] or [["inform", "advice", "stay_alert"]]
    elif kind == "I4 wrong priority":
        d = copy.deepcopy(next(o for o in outs if outcome(o) == "rescue_escalated" and o["goal"]["vulnerable"] != "নেই"))
        for a in acts_of(d):
            if a[0] == "escalate_rescue": a[2] = "normal"
    elif kind == "I5 capacity ignored":
        d = copy.deepcopy(pick["no_shelter_capacity"])
        for t in d["turns"]:
            t["acts"] = [["offer", "shelter", "কুতুবদিয়া আশ্রয়কেন্দ্র"] if a[0] == "escalate_officer" else a for a in t["acts"]]
    elif kind == "I6 unresolved":
        d = copy.deepcopy(pick["safe_advice"])
        for t in d["turns"]:
            t["acts"] = [a for a in t["acts"] if a[0] != "inform" or t["speaker"] == "U"] or [["greeting", None, None]]
    return d

kinds = ["I1 re-request", "I2 invalid shelter", "I3 rescue dropped", "I4 wrong priority", "I5 capacity ignored", "I6 unresolved"]
caught = 0
for k in kinds:
    errs = check_invariants(mutant(k)); caught += bool(errs)
    print(f"{k:<22} detected={bool(errs)}  {errs}")
print(f"\nMutants detected: {caught}/{len(kinds)}")


Writing mutation_test.py


In [4]:
%%writefile stress_test.py
"""Persona stress test (idea E) for the cyclone and flood response line.

User dialogue acts are realised as text in one of four registers. The bot's
keyword NLU parses the text back into acts; understanding failures trigger a
re-prompt, and two consecutive failures hand the caller to a human operator.

Every register has separate *dev* and *test* phrasings. NLU v1 uses only the
standard-Bangla dev phrasings; NLU v2 adds the other registers' dev phrasings
and fuzzy matching. Both are evaluated on the held-out *test* phrasings.
"""
import collections
import random
import re
import unicodedata

from selfplay import SystemFSM, UserSim, api_find_shelter, sample_scenario

ROMAN = {"কয়রা": "Koyra", "শ্যামনগর": "Shyamnagar", "কলাপাড়া": "Kalapara",
         "হাতিয়া": "Hatiya", "কুতুবদিয়া": "Kutubdia"}
BN_DIGITS = str.maketrans("0123456789", "০১২৩৪৫৬৭৮৯")

# (dev phrasings, test phrasings) per register and act value.
T = {
 "standard": {
  "status:rescue": (["আমাদের উদ্ধার দরকার"], ["আমাদের এখনই উদ্ধার করুন", "পানি ঘরে ঢুকছে, আমাদের বাঁচান"]),
  "status:shelter": (["আশ্রয়কেন্দ্রে যেতে চাই"], ["আশ্রয় দরকার", "থাকার একটা জায়গা দরকার"]),
  "status:safe": (["আমরা নিরাপদ আছি"], ["আমরা নিরাপদে আছি", "আমরা ঠিক আছি"]),
  "loc": (["{u} উপজেলায় আছি"], ["আমরা {u}তে আছি", "{u}"]),
  "loc_unknown": (["জায়গার নাম জানি না"], ["ঠিক কোথায় আছি জানি না"]),
  "landmark": (["{l}"], ["{l} আছি"]),
  "size": (["{n} জন"], ["আমরা মোট {n} জন"]),
  "vul:বয়স্ক": (["একজন বয়স্ক মানুষ আছেন"], ["বয়স্ক বাবা সাথে আছেন", "বৃদ্ধ মা সাথে আছেন"]),
  "vul:শিশু": (["ছোট শিশু আছে"], ["শিশু আছে একটা", "ছোট বাচ্চা আছে"]),
  "vul:অন্তঃসত্ত্বা": (["একজন অন্তঃসত্ত্বা মা আছেন"], ["স্ত্রী অন্তঃসত্ত্বা", "আমার স্ত্রী গর্ভবতী"]),
  "vul:প্রতিবন্ধী": (["একজন প্রতিবন্ধী মানুষ আছেন"], ["প্রতিবন্ধী ভাই আছে", "একজন হাঁটতে পারেন না"]),
  "vul:নেই": (["না, কেউ নেই"], ["কেউ নেই", "সবাই সুস্থ আছে"]),
  "affirm": (["হ্যাঁ, যাব"], ["হ্যাঁ যাব", "ঠিক আছে, আসছি"]),
  "negate": (["যেতে পারব না"], ["পারব না যেতে", "ওখানে যাওয়া সম্ভব না"]),
 },
 "banglish": {
  "status:rescue": (["amader uddhar dorkar"], ["uddhar koren please", "pani ghore dhukche, amader bachan"]),
  "status:shelter": (["ashroykendro te jete chai"], ["ashroy dorkar", "thakar jayga dorkar"]),
  "status:safe": (["amra nirapod achi"], ["amra nirapode", "amra thik achi"]),
  "loc": (["{r} upazila te achi"], ["amra {r} te", "{r}"]),
  "loc_unknown": (["jaygar nam jani na"], ["kothay achi jani na"]),
  "landmark": (["{lr}"], ["{lr} achi"]),
  "size": (["{n} jon"], ["total {n} jon"]),
  "vul:বয়স্ক": (["ekjon boyosko manush ache"], ["boyosko baba ache", "briddho ma ache"]),
  "vul:শিশু": (["choto shishu ache"], ["shishu ache", "choto baccha ache"]),
  "vul:অন্তঃসত্ত্বা": (["ekjon ontosotta ma ache"], ["wife ontosotta", "wife pregnant"]),
  "vul:প্রতিবন্ধী": (["ekjon protibondhi manush ache"], ["protibondhi bhai ache", "ekjon hatte pare na"]),
  "vul:নেই": (["na, keu nai"], ["keu nai", "sobai thik ache"]),
  "affirm": (["ha jabo"], ["ha", "ok ashchi"]),
  "negate": (["jete parbo na"], ["parbo na", "jawa possible na"]),
 },
 "sylheti_style": {   # approximate forms; to be replaced by native-speaker paraphrases
  "status:rescue": (["আমরারে উদ্ধার করইন"], ["উদ্ধার করইন তাড়াতাড়ি", "ফানি উঠি গেছে, আমরারে বাঁচাইন"]),
  "status:shelter": (["আশ্রয়কেন্দ্রো যাইতাম চাই"], ["আশ্রয় লাগবো", "থাকার জাগা লাগবো"]),
  "status:safe": (["আমরা ভালা আছি"], ["ভালা আছি আমরা", "কুনু সমস্যা নাই"]),
  "loc": (["{u}ত আছি"], ["আমরা {u}ত", "{u}"]),
  "loc_unknown": (["জাগার নাম জানি না"], ["কই আছি জানি না"]),
  "landmark": (["{l}"], ["{l} আছি"]),
  "size": (["{n} জন"], ["{n} জন মানুষ"]),
  "vul:বয়স্ক": (["এক বুড়া মানুষ আছইন"], ["বুড়া মা সাথে আছইন", "বয়স্ক বাবা আছইন"]),
  "vul:শিশু": (["ছোট বাচ্চা আছে"], ["বাচ্চা আছে", "ছোট ফুয়া আছে"]),
  "vul:অন্তঃসত্ত্বা": (["একজন অন্তঃসত্ত্বা মা আছইন"], ["বউ অন্তঃসত্ত্বা", "বউ গর্ভবতী"]),
  "vul:প্রতিবন্ধী": (["একজন প্রতিবন্ধী আছইন"], ["প্রতিবন্ধী ভাই আছইন", "একজন চলতে পারইন না"]),
  "vul:নেই": (["না, কেউ নাই"], ["কেউ নাই", "হকলে ভালা আছি"]),
  "affirm": (["অয়, যাইমু"], ["অয়", "ঠিক আছে, যাইমু"]),
  "negate": (["যাইতাম পারতাম নায়"], ["পারতাম নায়", "যাওয়া যাইতো নায়"]),
 },
 "noisy": {   # panic and ASR-like errors: repetition, dropped or misspelt characters
  "status:rescue": (["উদ্দার উদ্দার দরকার তাড়াতাড়ি"], ["উদ্ধর করেন প্লিজ", "উধার দরকার"]),
  "status:shelter": (["আশ্রয় কেন্দ্র যাব"], ["আশ্রয়ো দরকার", "আশ্রয় আশ্রয়"]),
  "status:safe": (["নিরাপদ নিরাপদ আছি"], ["নিরাপট আছি", "ঠিক আসি"]),
  "loc": (["{c} উপজেলা"], ["{c}তে", "{c}"]),
  "loc_unknown": (["জানি না জানি না"], ["জানিনা কোথায়"]),
  "landmark": (["{l}"], ["{l}"]),
  "size": (["{n}... {n} জন"], ["মোট {n}জন"]),
  "vul:বয়স্ক": (["বয়স্ক... বয়স্ক একজন"], ["বয়ষ্ক আছেন", "বয়স্ক আসেন"]),
  "vul:শিশু": (["শিশু আছে শিশু"], ["শিসু আছে", "শিশু আসে"]),
  "vul:অন্তঃসত্ত্বা": (["অন্তঃসত্তা মা"], ["অন্তসত্ত্বা", "অন্তঃসত্ত্বা মা আসে"]),
  "vul:প্রতিবন্ধী": (["প্রতিবন্দী আছে"], ["প্রতিবন্ধি", "প্রতিবন্ধী আসে"]),
  "vul:নেই": (["কেউ নাই কেউ নাই"], ["না নাই কেউ", "কেউ নেই"]),
  "affirm": (["হা যাব"], ["হ্যা যাবো", "হা হা"]),
  "negate": (["পারব না যেতে"], ["পারবো না", "যাওয়া যাবে না"]),
 },
}
LANDMARK_ROMAN = {"বড় মসজিদের পাশে": "boro masjid er pashe",
                  "প্রাইমারি স্কুলের কাছে": "primary school er kache",
                  "বাজারের পাশে": "bazar er pashe"}


def corrupt(word, rng):
    """Drop one internal character, simulating an ASR or typing error."""
    if len(word) < 4:
        return word
    i = rng.randrange(1, len(word) - 1)
    return word[:i] + word[i + 1:]


def realise(act, reg, split, rng):
    a, s, v = act
    pick = lambda key: rng.choice(T[reg][key][0 if split == "dev" else 1])
    if a == "inform" and s == "status":
        return pick(f"status:{v}")
    if a == "inform" and s == "location":
        if v == "অজানা":
            return pick("loc_unknown")
        return pick("loc").format(u=v, r=ROMAN[v], c=corrupt(v, rng))
    if a == "inform" and s == "landmark":
        return pick("landmark").format(l=v, lr=LANDMARK_ROMAN[v])
    if a == "inform" and s == "household_size":
        n = str(v) if reg == "banglish" else str(v).translate(BN_DIGITS)
        return pick("size").format(n=n)
    if a == "inform" and s == "vulnerable":
        return pick(f"vul:{v}")
    if a in ("affirm", "negate"):
        return pick(a)
    return ""


# --------------------------------------------------------------- NLU --
def lexicon(version):
    lex = {
        "status": {"rescue": ["উদ্ধার"], "shelter": ["আশ্রয়"], "safe": ["নিরাপদ"]},
        "unknown": ["জানি না"],
        "landmark": ["পাশে", "কাছে"],
        "vulnerable": {"বয়স্ক": ["বয়স্ক"], "শিশু": ["শিশু"], "অন্তঃসত্ত্বা": ["অন্তঃসত্ত্বা"],
                       "প্রতিবন্ধী": ["প্রতিবন্ধী"], "নেই": ["কেউ নেই"]},
        "affirm": ["হ্যাঁ"], "negate": ["পারব না"],
        "places": {u: [u] for u in ROMAN},
    }
    if version == 2:     # additions derived only from the other registers' DEV phrasings
        lex["status"]["rescue"] += ["uddhar", "উদ্দার"]
        lex["status"]["shelter"] += ["ashroy"]
        lex["status"]["safe"] += ["nirapod", "ভালা"]
        lex["unknown"] += ["jani na"]
        lex["landmark"] += ["pashe", "kache"]
        lex["vulnerable"]["বয়স্ক"] += ["boyosko", "বুড়া"]
        lex["vulnerable"]["শিশু"] += ["shishu", "বাচ্চা"]
        lex["vulnerable"]["অন্তঃসত্ত্বা"] += ["ontosotta"]
        lex["vulnerable"]["প্রতিবন্ধী"] += ["protibondhi"]
        lex["vulnerable"]["নেই"] += ["keu nai", "কেউ নাই"]
        lex["affirm"] += ["ha", "অয়", "হা"]
        lex["negate"] += ["parbo na", "পারতাম নায়"]
        for u, r in ROMAN.items():
            lex["places"][u].append(r.lower())
    return lex


def _lev(a, b):
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]


def _has(text, key, fuzzy):
    toks = re.findall(r"[^\s,।.!?…]+", text)
    if " " in key or len(key) <= 3:                  # phrases and short words: exact
        return key in text if " " in key else key in toks
    if key in text:
        return True
    return fuzzy and any(abs(len(t) - len(key)) <= 1 and _lev(t, key) <= 1 for t in toks)


def parse(text, expected, version):
    """Return the dialogue acts understood from `text`."""
    text = unicodedata.normalize("NFC", text).lower()
    lex, fuzzy, acts = lexicon(version), version == 2, []
    if expected == "offer":
        if any(_has(text, k, fuzzy) for k in lex["negate"]):
            return [("negate", "travel", None)]
        if any(_has(text, k, fuzzy) for k in lex["affirm"]):
            return [("affirm", None, None)]
        return []
    for val, keys in lex["status"].items():
        if any(_has(text, k, fuzzy) for k in keys):
            acts.append(("inform", "status", val)); break
    if any(_has(text, k, fuzzy) for k in lex["unknown"]):
        acts.append(("inform", "location", "অজানা"))
    else:
        for u, keys in lex["places"].items():
            if any(_has(text, k.lower(), fuzzy) for k in keys):
                acts.append(("inform", "location", u)); break
    if expected == "landmark" and any(_has(text, k, fuzzy) for k in lex["landmark"]):
        acts.append(("inform", "landmark", text))
    m = re.search(r"[0-9০-৯]+", text)
    if m:
        acts.append(("inform", "household_size", int(m.group().translate(str.maketrans("০১২৩৪৫৬৭৮৯", "0123456789")))))
    for val, keys in lex["vulnerable"].items():
        if any(_has(text, k, fuzzy) for k in keys):
            acts.append(("inform", "vulnerable", val)); break
    return acts


# ------------------------------------------------------------- dialogue --
def expected_outcome(g):
    if g["status"] == "safe":
        return ("safe_advice",)
    pri = "high" if g["vulnerable"] != "নেই" else "normal"
    if g["status"] == "rescue":
        return ("rescue", g["location"], g["household_size"], pri)
    if g["location"] == "অজানা":
        return ("officer_location_unknown",)
    shelter = api_find_shelter(g["location"], g["household_size"])
    if shelter is None:
        return ("officer_no_capacity",)
    return ("shelter", shelter) if g["can_travel"] else ("rescue", g["location"], g["household_size"], pri)


def run_dialogue(rng, reg, version, split="test", max_turns=24):
    goal, profile = sample_scenario(rng)
    user, system = UserSim(goal, profile, rng), SystemFSM()
    sys_acts, expected, fails, turns = [("request", "status", None)], "status", 0, 0
    while turns < max_turns:
        turns += 1
        u_acts = user.respond(sys_acts) if sys_acts[-1][0] != "repeat" else \
                 ([("inform", expected, goal[expected])] if expected != "offer"
                  else ([("affirm", None, None)] if goal["can_travel"] else [("negate", "travel", None)]))
        text = " , ".join(realise(a, reg, split, rng) for a in u_acts if a[0] in ("inform", "affirm", "negate"))
        if u_acts[-1][0] == "bye":
            break
        parsed = parse(text, expected, version)
        got_expected = any(a[0] in ("affirm", "negate") for a in parsed) if expected == "offer" \
                       else any(a[1] == expected for a in parsed)
        if not got_expected:
            fails += 1
            if fails >= 2:
                return "handoff", turns
            for a in parsed:                       # keep other slots understood, if still empty
                if a[0] == "inform" and a[1] not in system.state:
                    system.state[a[1]] = a[2]
            sys_acts = [("repeat", expected, None)]
            continue
        fails = 0
        # fill only the requested slot and slots still empty; never overwrite
        parsed = [a for a in parsed if a[0] != "inform" or a[1] == expected or a[1] not in system.state]
        sys_acts = system.act(parsed)
        a, s, v = sys_acts[-1]
        if a == "request":
            expected = s
        elif a == "offer":
            expected = "offer"
        else:
            break
    st, (a, s, v) = system.state, sys_acts[-1]
    if a == "inform":
        got = ("safe_advice",)
    elif a == "escalate_rescue":
        got = ("rescue", st.get("location"), st.get("household_size"), v)
    elif a == "escalate_officer":
        got = ("officer_location_unknown",) if v == "location_unknown" else ("officer_no_capacity",)
    elif a == "notify_success":
        got = ("shelter", v)
    else:
        return "incomplete", turns
    return ("correct" if got == expected_outcome(goal) else "wrong"), turns


def evaluate(version, split="test", n=400, seed=2026):
    rows = {}
    for reg in T:
        rng = random.Random(seed)
        res = [run_dialogue(rng, reg, version, split) for _ in range(n)]
        c = collections.Counter(r for r, _ in res)
        rows[reg] = {k: c[k] / n for k in ("correct", "handoff", "wrong", "incomplete")}
        rows[reg]["turns"] = sum(t for _, t in res) / n
    return rows


if __name__ == "__main__":
    for split in ("dev", "test"):
        for version in (1, 2):
            rows = evaluate(version, split)
            print(f"\nNLU v{version} on {split.upper()} phrasings (400 dialogues per persona)")
            print(f"  {'persona':<15}{'correct':>9}{'handoff':>9}{'wrong':>8}{'incompl':>9}{'turns':>7}")
            for reg, r in rows.items():
                print(f"  {reg:<15}{r['correct']:>9.1%}{r['handoff']:>9.1%}{r['wrong']:>8.1%}{r['incomplete']:>9.1%}{r['turns']:>7.1f}")
            vals = [r["correct"] for r in rows.values()]
            print(f"  fairness gap (max - min correct): {max(vals) - min(vals):.1%}")


Writing stress_test.py


## 2. Stage 1: self-play, invariants and mutation testing

In [5]:
!python selfplay.py
!python evaluate_outlines.py
!python mutation_test.py

wrote 1000 outlines


Invariant violations: 0 across 1000 outlines

Outcome distribution:
  safe_advice                     331  (33.1%)
  rescue_escalated                305  (30.5%)
  shelter_confirmed               193  (19.3%)
  rescue_after_shelter_refusal     75  (7.5%)
  no_shelter_capacity              62  (6.2%)
  location_unknown                 34  (3.4%)

Turns per dialogue: mean 7.08, min 4, max 12

Diversity (self-play vs scripted):
  unique_transitions_per_turn    0.0133   0.0009
  unique_subdialogues_k3         0.0385   0.0010
  unique_outlines_ratio          0.0710   0.0010


I1 re-request          detected=True  ['I1 re-requested location']
I2 invalid shelter     detected=True  ['I2 invalid shelter offer']
I3 rescue dropped      detected=True  ['I3 rescue need not escalated']
I4 wrong priority      detected=True  ['I4 wrong priority']
I5 capacity ignored    detected=True  ['I2 invalid shelter offer', 'I5 capacity handling wrong', 'I6 caller left unresolved']
I6 unresolved          detected=True  ['I6 caller left unresolved']

Mutants detected: 6/6


## 3. Stage 2: persona stress test (v1 vs v2, dev and test phrasings)

In [6]:
!python stress_test.py


NLU v1 on DEV phrasings (400 dialogues per persona)
  persona          correct  handoff   wrong  incompl  turns
  standard          100.0%     0.0%    0.0%     0.0%    2.4
  banglish            0.0%   100.0%    0.0%     0.0%    2.0
  sylheti_style      17.8%    82.2%    0.0%     0.0%    3.2
  noisy              38.0%    62.0%    0.0%     0.0%    2.0
  fairness gap (max - min correct): 100.0%



NLU v2 on DEV phrasings (400 dialogues per persona)
  persona          correct  handoff   wrong  incompl  turns
  standard          100.0%     0.0%    0.0%     0.0%    2.4
  banglish          100.0%     0.0%    0.0%     0.0%    2.4
  sylheti_style     100.0%     0.0%    0.0%     0.0%    2.4
  noisy              97.5%     2.5%    0.0%     0.0%    2.4
  fairness gap (max - min correct): 2.5%



NLU v1 on TEST phrasings (400 dialogues per persona)
  persona          correct  handoff   wrong  incompl  turns
  standard           63.2%    36.8%    0.0%     0.0%    2.8
  banglish            0.0%   100.0%    0.0%     0.0%    2.0
  sylheti_style      13.8%    86.2%    0.0%     0.0%    3.1
  noisy               0.0%   100.0%    0.0%     0.0%    2.4
  fairness gap (max - min correct): 63.2%



NLU v2 on TEST phrasings (400 dialogues per persona)
  persona          correct  handoff   wrong  incompl  turns
  standard           64.2%    35.8%    0.0%     0.0%    2.7
  banglish           63.2%    36.8%    0.0%     0.0%    2.8
  sylheti_style      64.2%    33.5%    2.2%     0.0%    2.7
  noisy              57.8%    42.2%    0.0%     0.0%    2.7
  fairness gap (max - min correct): 6.5%


## 4. Automatic check of every claim in Answer 3

In [7]:
import json, subprocess, hashlib, importlib
import evaluate_outlines as ev, stress_test as st
importlib.reload(ev); importlib.reload(st)

outlines = [json.loads(l) for l in open("outlines.jsonl", encoding="utf-8")]
violations = sum(len(ev.check_invariants(d)) for d in outlines)
mut = subprocess.run(["python", "mutation_test.py"], capture_output=True, text=True).stdout
mut_line = [l for l in mut.splitlines() if l.startswith("Mutants detected")][0]
v1, v2 = st.evaluate(1), st.evaluate(2)
gap = lambda rows: max(r["correct"] for r in rows.values()) - min(r["correct"] for r in rows.values())
misread = st.parse("থাকার জাগা লাগবো , হকলে ভালা আছি", "status", 2)

claims = [
    ("1,000 synthetic dialogues generated",            len(outlines) == 1000,                    f"{len(outlines)} dialogues"),
    ("No violations of six safety invariants",        violations == 0,                          f"{violations} violations"),
    ("Checker validated by mutation testing (6/6)",   mut_line.endswith("6/6"),                 mut_line),
    ("v1 resolves 63.2% of standard-Bangla callers",  round(v1["standard"]["correct"], 3) == 0.632, f"{v1['standard']['correct']:.1%}"),
    ("v1 resolves no Banglish callers",               v1["banglish"]["correct"] == 0,           f"{v1['banglish']['correct']:.1%}"),
    ("v1 resolves no panicked (noisy) callers",       v1["noisy"]["correct"] == 0,              f"{v1['noisy']['correct']:.1%}"),
    ("v1 fairness gap is 63.2 points",                round(gap(v1), 3) == 0.632,               f"{gap(v1)*100:.1f} pts"),
    ("v2 cuts the fairness gap to 6.5 points",        round(gap(v2), 3) == 0.065,               f"{gap(v2)*100:.1f} pts"),
    ("v2 misreads a shelter request as 'safe'",       misread == [("inform", "status", "safe")], str(misread)),
]
print(f"{'Claim in Answer 3':<46}{'Verified':<10}Computed value")
print("-" * 86)
for claim, ok, value in claims:
    print(f"{claim:<46}{'YES' if ok else 'NO':<10}{value}")
print("-" * 86)
print(f"All claims verified: {all(ok for _, ok, _ in claims)}")
print("SHA-256 of outlines.jsonl:", hashlib.sha256(open("outlines.jsonl", "rb").read()).hexdigest())

Claim in Answer 3                             Verified  Computed value
--------------------------------------------------------------------------------------
1,000 synthetic dialogues generated           YES       1000 dialogues
No violations of six safety invariants        YES       0 violations
Checker validated by mutation testing (6/6)   YES       Mutants detected: 6/6
v1 resolves 63.2% of standard-Bangla callers  YES       63.2%
v1 resolves no Banglish callers               YES       0.0%
v1 resolves no panicked (noisy) callers       YES       0.0%
v1 fairness gap is 63.2 points                YES       63.2 pts
v2 cuts the fairness gap to 6.5 points        YES       6.5 pts
v2 misreads a shelter request as 'safe'       YES       [('inform', 'status', 'safe')]
--------------------------------------------------------------------------------------
All claims verified: True
SHA-256 of outlines.jsonl: bb1a9bb8bf87d1a0dd0aec4fea3f11eceefca103992fd94f668882518da09276


## 5. Read a generated dialogue
Change the outcome below (`rescue_escalated`, `shelter_confirmed`, `rescue_after_shelter_refusal`, `no_shelter_capacity`, `location_unknown`, `safe_advice`) to inspect other flows.

In [8]:
WANTED = "rescue_after_shelter_refusal"
d = next(o for o in outlines if ev.outcome(o) == WANTED)
print("Goal:", d["goal"], "\n")
for t in d["turns"]:
    acts = "; ".join(f"{a}({s}={v})" if s else a for a, s, v in t["acts"])
    print(("SYSTEM" if t["speaker"] == "S" else "CALLER").ljust(7), acts)

Goal: {'status': 'shelter', 'location': 'হাতিয়া', 'household_size': 8, 'vulnerable': 'শিশু', 'can_travel': False} 

SYSTEM  greeting; request(status=None)
CALLER  inform(status=shelter); inform(location=হাতিয়া); inform(vulnerable=শিশু)
SYSTEM  request(household_size=None)
CALLER  inform(household_size=8)
SYSTEM  offer(shelter=হাতিয়া সদর আশ্রয়কেন্দ্র)
CALLER  negate(travel=None)
SYSTEM  escalate_rescue(priority=high)
CALLER  bye
